In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files
uploaded = files.upload() # click Browse -> select application_test.xlsx from your laptop

Saving application_test (1).xlsx to application_test (1).xlsx


In [9]:
import os
print(os.listdir())

['.config', 'drive', '.ipynb_checkpoints', 'application_test .xlsx', 'sample_data']


In [10]:
import pandas as pd
df = pd.read_excel("application_test .xlsx")
df.head()

,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,AMT_REQ_CREDIT_BUREAU_YEAR.1,Unnamed: 122
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,2.0000
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0.0,0.0,0.0,0.0,0.0,3.0,NaN,2.0000
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0.0,0.0,0.0,0.0,1.0,4.0,NaN,2.0000
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0.0,0.0,0.0,0.0,0.0,3.0,NaN,26.0000
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,516740.4356


In [12]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Select ONLY numbers and fill missing values
df_numeric = df.select_dtypes(include='number')
df_numeric = df_numeric.fillna(0)

print("Numeric columns:", df_numeric.shape)

# 2. Scale it - important for clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_numeric)

# 3. KMeans
km = KMeans(n_clusters=3, random_state=42)
labels = km.fit_predict(X_scaled)

print("number of unique clusters", np.unique(labels))
print("total number of rows", len(labels))

# 4. Add to main df
df['k means cluster'] = labels

df['k means cluster'].value_counts()

Numeric columns: (48744, 107)
number of unique clusters [0 1 2]
total number of rows 48744


,count
k means cluster,
0,29860
1,16518
2,2366


In [ ]:
import pickle
with open("kmeans.pkl",'wb') as file:
  pickle.dump(km,file)

In [14]:
from sklearn.cluster import AgglomerativeClustering

import numpy as np
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), 5000, replace=False)
X_sample = X_scaled[sample_idx]

ag = AgglomerativeClustering(n_clusters=3)
labels_ag_sample = ag.fit_predict(X_sample)

print("number of unique clusters", np.unique(labels_ag_sample))
print("total number of rows in sample", len(labels_ag_sample))
import pandas as pd
print(pd.Series(labels_ag_sample).value_counts())



number of unique clusters [0 1 2]
total number of rows in sample 5000
1    2460
2    1922
0     618
Name: count, dtype: int64


In [16]:
from sklearn.cluster import DBSCAN
import numpy as np


X_small = X_scaled[:, :3]

db = DBSCAN(eps=0.5, min_samples=5)
labels_db = db.fit_predict(X_small)

print("number of unique clusters", np.unique(labels_db))
print("total number of rows", len(labels_db))


df['DBSCAN cluster'] = labels_db

df['DBSCAN cluster'].value_counts()

number of unique clusters [-1  0  1  2  3  4  5  6  7]
total number of rows 48744


,count
DBSCAN cluster,
0,34649
2,9481
1,3928
3,525
-1,103
4,29
7,18
5,6
6,5


In [17]:
from sklearn.cluster import MeanShift
import numpy as np
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), 2000, replace=False)
X_sample = X_scaled[sample_idx][:, :3]

ms = MeanShift()
labels_ms = ms.fit_predict(X_sample)

print("number of unique clusters", np.unique(labels_ms))
print("total number of rows", len(labels_ms))
print("value counts:\n", np.unique(labels_ms, return_counts=True))

number of unique clusters [0 1 2 3 4]
total number of rows 2000
value counts:
 (array([0, 1, 2, 3, 4]), array([1952,    8,   17,   22,    1]))


In [ ]:
from sklearn.cluster import MeanShift

ms = MeanShift()

ms.fit_predict(X)

print("number of unique clusters", np.unique(ms.labels_))

print("total number of rows", len(ms.labels_))

df['MeanShift cluster'] = ms.labels_

df['MeanShift cluster'].value_counts()

In [18]:
pip install kmodes

In [20]:
!pip install kmodes -q
from kmodes.kmodes import KModes
import numpy as np

# These are the ACTUAL text columns in application_test.xlsx
cat_cols = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR',
            'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE',
            'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS']

# Keep only columns that exist and have text
cat_cols = [c for c in cat_cols if c in df.columns]

X_cat = df[cat_cols].astype(str).fillna('Unknown')

print("Using columns:", cat_cols)
print("Shape:", X_cat.shape)
print(X_cat.head(3))

kmod = KModes(n_clusters=5, init='Huang', n_init=5, verbose=0, random_state=42)
kmod_labels = kmod.fit_predict(X_cat)

print("number of unique clusters", np.unique(kmod_labels))
print("total number of rows", len(kmod_labels))

df['KModes cluster'] = kmod_labels
df['KModes cluster'].value_counts()

Using columns: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS']
Shape: (48744, 8)
  NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY NAME_TYPE_SUITE  \
0         Cash loans           F            N               Y   Unaccompanied   
1         Cash loans           M            N               Y   Unaccompanied   
2         Cash loans           M            Y               Y             nan   

  NAME_INCOME_TYPE            NAME_EDUCATION_TYPE NAME_FAMILY_STATUS  
0          Working               Higher education            Married  
1          Working  Secondary / secondary special            Married  
2          Working               Higher education            Married  
number of unique clusters [0 1 2 3 4]
total number of rows 48744


,count
KModes cluster,
1,15587
0,11763
2,11464
4,7537
3,2393


In [22]:
!pip install kmodes -q
from kmodes.kprototypes import KPrototypes
import numpy as np

# 1. For application_test.xlsx - correct columns
num_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'CNT_CHILDREN']
cat_cols = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']

# Keep only columns that exist
num_cols = [c for c in num_cols if c in df.columns]
cat_cols = [c for c in cat_cols if c in df.columns]

print("Numeric:", num_cols)
print("Categorical:", cat_cols)

X_kp = df[num_cols + cat_cols].copy()
X_kp[cat_cols] = X_kp[cat_cols].astype(str).fillna('Unknown')
X_kp[num_cols] = X_kp[num_cols].fillna(0)

# For KPrototypes, need to tell which columns are categorical by INDEX
# Example: if 3 numeric + 4 categorical, categorical indices are [3,4,5,6]
cat_indices = list(range(len(num_cols), len(num_cols)+len(cat_cols)))
print("Categorical indices:", cat_indices)

kproto = KPrototypes(n_clusters=5, init='Huang', random_state=42)
# Use sample of 5000 for speed, otherwise very slow
X_sample = X_kp.sample(n=5000, random_state=42).to_numpy()

labels = kproto.fit_predict(X_sample, categorical=cat_indices)

print("number of unique clusters", np.unique(labels))
print("total number of rows", len(labels))

Numeric: ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'CNT_CHILDREN']
Categorical: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']
Categorical indices: [3, 4, 5, 6]
number of unique clusters [0 1 2 3 4]
total number of rows 5000
